# Bai existing candidate re-evaluation

Re-evaluates the first real trained Bai adapter under the corrected shared SFT/eval prompt contract. No retraining, no release creation, unchanged promotion gate. If rejected, produces failure clusters and a pending human-review queue. It also emits an apples-to-apples candidate comparison on the same frozen holdout; historical Version 1 predictions are included automatically when present in the downloaded Kaggle output. Final cells package portable evidence and adapter ZIPs with pinned hashes for handoff.


In [ ]:
import os, sys, subprocess, json, shutil, zipfile
from pathlib import Path
print('Python', sys.version)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.51,<5', 'peft>=0.15,<1', 'datasets>=3,<5', 'accelerate', 'bitsandbytes', 'sentencepiece', 'huggingface-hub', 'kagglehub'], check=True)
import kagglehub
print('kagglehub ready')


In [ ]:
ROOT=Path('/kaggle/working/tamdeshevle')
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth','1','https://github.com/eneonstudio-dev/tamdeshevle.git',str(ROOT)],check=True)
print('repo', subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())


In [ ]:
SOURCE='eneonstii/notebook07f42bd563/versions/1'
downloaded=Path(kagglehub.notebook_output_download(SOURCE))
print('downloaded', downloaded)
zips=list(downloaded.rglob('bai-candidate-experiment.zip'))
if not zips: raise FileNotFoundError('bai-candidate-experiment.zip not found in Version 1 output')
candidate_root=Path('/kaggle/working/first-bai-candidate')
if candidate_root.exists(): shutil.rmtree(candidate_root)
candidate_root.mkdir(parents=True)
with zipfile.ZipFile(zips[0]) as z: z.extractall(candidate_root)
adapter=candidate_root/'adapter'
if not adapter.is_dir(): raise FileNotFoundError(f'adapter missing after extract: {adapter}')
print('adapter', adapter)


In [ ]:
seed=Path('/kaggle/working/bai-reeval-seed')
if seed.exists(): shutil.rmtree(seed)
subprocess.run(['node',str(ROOT/'teacher-lab/training/deterministic-seed.mjs'),str(seed)],cwd=ROOT,check=True)
eval_gold=seed/'eval-gold.jsonl'
print('heldout rows', sum(1 for line in eval_gold.open(encoding='utf-8') if line.strip()))


In [ ]:
out=Path('/kaggle/working/bai-candidate-reeval')
if out.exists(): shutil.rmtree(out)
cmd=[sys.executable,str(ROOT/'teacher-lab/training/reevaluate_candidate.py'),'--eval-gold',str(eval_gold),'--candidate-adapter',str(adapter),'--out',str(out)]
result=subprocess.run(cmd,cwd=ROOT)
print('reeval exit', result.returncode)
manifest=json.loads((out/'reeval-manifest.json').read_text(encoding='utf-8'))
print(json.dumps(manifest,ensure_ascii=False,indent=2))


In [ ]:
pred=out/'candidate-predictions.jsonl'
if not pred.is_file(): raise FileNotFoundError('candidate predictions missing; re-evaluation did not reach inference output')
fail=out/'failure-analysis'
subprocess.run([sys.executable,str(ROOT/'teacher-lab/training/analyze_candidate_failures.py'),'--eval-gold',str(eval_gold),'--predictions',str(pred),'--out-dir',str(fail)],cwd=ROOT,check=True)
print((fail/'failure-summary.json').read_text(encoding='utf-8'))


In [ ]:
comparison=out/'comparison'
runs=[]
historical_root=None
for historical_candidate in sorted(downloaded.rglob('candidate-predictions.jsonl')):
    root=historical_candidate.parent
    required=[root/'baseline-predictions.jsonl',root/'metrics'/'baseline.json',root/'metrics'/'candidate.json']
    if all(p.is_file() for p in required):
        historical_root=root; break
if historical_root is not None:
    print('historical comparison root', historical_root)
    runs += ['--run','historical-baseline',str(historical_root/'baseline-predictions.jsonl'),str(historical_root/'metrics'/'baseline.json')]
    runs += ['--run','historical-candidate',str(historical_root/'candidate-predictions.jsonl'),str(historical_root/'metrics'/'candidate.json')]
else:
    print('Historical prediction files not present in Version 1 output; comparing corrected runs only')
runs += ['--run','corrected-baseline',str(out/'baseline-predictions.jsonl'),str(out/'metrics'/'baseline.json')]
runs += ['--run','corrected-candidate',str(out/'candidate-predictions.jsonl'),str(out/'metrics'/'candidate.json')]
compare_cmd=[sys.executable,str(ROOT/'teacher-lab/training/compare_candidate_runs.py'),'--eval-gold',str(eval_gold),*runs,'--out-dir',str(comparison)]
subprocess.run(compare_cmd,cwd=ROOT,check=True)
print((comparison/'candidate-comparison.md').read_text(encoding='utf-8'))
print('Artifacts:')
for p in sorted(out.rglob('*')):
    if p.is_file(): print(p)


In [ ]:
handoff_prefix=Path('/kaggle/working/bai_candidate_reeval')
pack_cmd=[sys.executable,str(ROOT/'teacher-lab/training/package_reeval_artifact.py'),'--reeval-dir',str(out),'--eval-gold',str(eval_gold),'--candidate-adapter',str(adapter),'--out-prefix',str(handoff_prefix),'--source-ref',SOURCE]
subprocess.run(pack_cmd,cwd=ROOT,check=True)
handoff_file=Path(str(handoff_prefix)+'-handoff.json')
handoff=json.loads(handoff_file.read_text(encoding='utf-8'))
print(json.dumps(handoff,ensure_ascii=False,indent=2))
print('Download these handoff artifacts:')
print(handoff['evidence_zip'])
print(handoff['adapter_zip'])
print(handoff_file)
